In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import List, Dict, Optional
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import load_dataset

/Users/alyssagardiner/src/sae_trainer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
root = Path.cwd().resolve()
root = root.parent
sys.path.insert(0, str(root))

from sae_trainer.train import training_wrapper, get_model, get_dataloader, collect_activations, 

In [ ]:
cfg = {
    # Model
    'model_name': 'gpt2',
    #d_in: 768 # inferred from model
    'expansion_factor': 8,
    'normalize_decoder': True,
    'collection_batch_size': 8,

    # Data
    'dataset_name': 'wikitext',

    # Logging
    'use_wandb': False,
    'wandb_project': 'sae-trainer',

    # Training
    'num_epochs': 5,
    'sae_batch_size': 2048,
    'lr': 3.0e-4,
    'weight_decay': 1.0e-4,
    'lambda_l1': 1.0e-2,
    'lambda_kl': 1.0e-3,
    'target_firing_rate': 0.02,
    'mass_frac_threshold': 0.001,
}

In [ ]:
save_mode = False

device = "cuda" if torch.cuda.is_available() else "cpu"

model, tokenizer, collector_class = get_model(cfg, device)

loader = get_dataloader(cfg, tokenizer)

collector = collector_class(model=model)
target_layers = collector.get_layers()
collector.register()

accum = collect_activations(loader, collector, target_layers, device, max_batches=50)
    
mass_frac_threshold=cfg.mass_frac_threshold

saes = {}
histories = {}
train_loaders = {}
val_loaders = {}

for target_layer in target_layers:
    print(f"Training SAE for layer {target_layer}")
    sae, history, train_loader, val_loader = training_wrapper(cfg, accum, target_layer, device, mass_frac_threshold, save_mode=save_mode, show_curves=True)
    saes[target_layer] = sae
    histories[target_layer] = history
    train_loaders[target_layer] = train_loader
    val_loaders[target_layer] = val_loader